In [1]:
# Check whether Sheet2 TP # (unsequenced cfdna samples) match Sheet3 TP Number (Cases having Enrich poitive samples)

import pandas as pd
import re
from pathlib import Path

# ===============================
# Input Excel file
# ===============================
xlsx_path = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/Unsequenced_data.xlsx")

# ===============================
# Output files
# ===============================
out_dir = xlsx_path.parent / "TP_match_check"
out_dir.mkdir(exist_ok=True)

match_out = out_dir / "Sheet2_TP_matches_Sheet3_TPNumber.csv"
not_found_out = out_dir / "Sheet2_TP_not_found_in_Sheet3.csv"
sheet3_long_out = out_dir / "Sheet3_extracted_TPNumber_long.csv"

# ===============================
# Helper functions
# ===============================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def extract_full_tp(x):
    """
    Extract TP with optional timepoint number.
    Example:
      TP24-M115 (6) PORT -> TP24-M115(6)
      TP21-M500(5)      -> TP21-M500(5)
      TP24-M292         -> TP24-M292
    """
    x = clean_text(x).upper()

    m = re.search(r"(TP\d{2}-M\d+)\s*(?:\(\s*(\d+)\s*\))?", x)
    if not m:
        return ""

    base = m.group(1)
    tp_num = m.group(2)

    if tp_num:
        return f"{base}({tp_num})"
    else:
        return base


def extract_base_tp(x):
    """
    Extract only base TP ID.
    Example:
      TP24-M115 (6) PORT -> TP24-M115
      TP21-M500(5)      -> TP21-M500
    """
    x = clean_text(x).upper()
    m = re.search(r"TP\d{2}-M\d+", x)
    return m.group(0) if m else ""


# ===============================
# Read sheets
# ===============================
sheet2 = pd.read_excel(xlsx_path, sheet_name="Sheet2", dtype=str)
sheet3 = pd.read_excel(xlsx_path, sheet_name="Sheet3", dtype=str)

sheet2.columns = [clean_text(c) for c in sheet2.columns]
sheet3.columns = [clean_text(c) for c in sheet3.columns]

# ===============================
# Identify relevant columns
# ===============================
sheet2_tp_col = "TP #"

if sheet2_tp_col not in sheet2.columns:
    raise ValueError(f"Could not find '{sheet2_tp_col}' in Sheet2. Found columns: {list(sheet2.columns)}")

# Because Sheet3 has repeated TP Number columns, pandas may name them:
# TP Number, TP Number.1, TP Number.2, etc.
sheet3_tp_cols = [
    c for c in sheet3.columns
    if re.match(r"^TP Number(\.\d+)?$", c)
]

if not sheet3_tp_cols:
    raise ValueError(f"No TP Number columns found in Sheet3. Found columns: {list(sheet3.columns)}")

print(f"Found {len(sheet3_tp_cols)} TP Number columns in Sheet3:")
for c in sheet3_tp_cols:
    print(f"  - {c}")

# ===============================
# Prepare Sheet2 TP table
# ===============================
sheet2_tp = sheet2.copy()
sheet2_tp["Sheet2_TP_raw"] = sheet2_tp[sheet2_tp_col].apply(clean_text)
sheet2_tp["Sheet2_TP_full_norm"] = sheet2_tp["Sheet2_TP_raw"].apply(extract_full_tp)
sheet2_tp["Sheet2_TP_base"] = sheet2_tp["Sheet2_TP_raw"].apply(extract_base_tp)

sheet2_tp = sheet2_tp[sheet2_tp["Sheet2_TP_base"] != ""].copy()

# ===============================
# Convert Sheet3 TP Number columns to long format
# ===============================
records = []

for idx, row in sheet3.iterrows():
    for col in sheet3_tp_cols:
        raw_tp = clean_text(row[col])

        if raw_tp == "":
            continue

        full_norm = extract_full_tp(raw_tp)
        base_tp = extract_base_tp(raw_tp)

        if base_tp == "":
            continue

        records.append({
            "Sheet3_Row_Index": idx,
            "Sheet3_TP_Column": col,
            "Sheet3_TP_raw": raw_tp,
            "Sheet3_TP_full_norm": full_norm,
            "Sheet3_TP_base": base_tp,
            "Sheet3_REDCAP": clean_text(row.get("REDCAP#", "")),
            "Sheet3_BIOS": clean_text(row.get("BIOS #", "")),
            "Sheet3_Histology": clean_text(row.get("IDC/ILC", "")),
        })

sheet3_tp_long = pd.DataFrame(records)

sheet3_tp_long.to_csv(sheet3_long_out, index=False)

print(f"\nExtracted {len(sheet3_tp_long)} TP Number entries from Sheet3")
print(f"Saved Sheet3 long TP table: {sheet3_long_out}")

# ===============================
# Match Sheet2 against Sheet3
# ===============================
match_records = []
not_found_records = []

for _, s2 in sheet2_tp.iterrows():

    s2_raw = s2["Sheet2_TP_raw"]
    s2_full = s2["Sheet2_TP_full_norm"]
    s2_base = s2["Sheet2_TP_base"]

    # First: full TP/timepoint match
    full_matches = sheet3_tp_long[
        sheet3_tp_long["Sheet3_TP_full_norm"] == s2_full
    ].copy()

    # Second: base TP match
    base_matches = sheet3_tp_long[
        sheet3_tp_long["Sheet3_TP_base"] == s2_base
    ].copy()

    if len(full_matches) > 0:
        for _, m in full_matches.iterrows():
            match_records.append({
                "Match_Type": "FULL_TP_MATCH",
                "Sheet2_TP_raw": s2_raw,
                "Sheet2_TP_full_norm": s2_full,
                "Sheet2_TP_base": s2_base,
                "Sheet2_Histology": clean_text(s2.get("Histology", "")),
                "Sheet3_TP_raw": m["Sheet3_TP_raw"],
                "Sheet3_TP_full_norm": m["Sheet3_TP_full_norm"],
                "Sheet3_TP_base": m["Sheet3_TP_base"],
                "Sheet3_Row_Index": m["Sheet3_Row_Index"],
                "Sheet3_TP_Column": m["Sheet3_TP_Column"],
                "Sheet3_REDCAP": m["Sheet3_REDCAP"],
                "Sheet3_BIOS": m["Sheet3_BIOS"],
                "Sheet3_Histology": m["Sheet3_Histology"],
            })

    elif len(base_matches) > 0:
        for _, m in base_matches.iterrows():
            match_records.append({
                "Match_Type": "BASE_TP_MATCH_ONLY",
                "Sheet2_TP_raw": s2_raw,
                "Sheet2_TP_full_norm": s2_full,
                "Sheet2_TP_base": s2_base,
                "Sheet2_Histology": clean_text(s2.get("Histology", "")),
                "Sheet3_TP_raw": m["Sheet3_TP_raw"],
                "Sheet3_TP_full_norm": m["Sheet3_TP_full_norm"],
                "Sheet3_TP_base": m["Sheet3_TP_base"],
                "Sheet3_Row_Index": m["Sheet3_Row_Index"],
                "Sheet3_TP_Column": m["Sheet3_TP_Column"],
                "Sheet3_REDCAP": m["Sheet3_REDCAP"],
                "Sheet3_BIOS": m["Sheet3_BIOS"],
                "Sheet3_Histology": m["Sheet3_Histology"],
            })

    else:
        not_found_records.append({
            "Sheet2_TP_raw": s2_raw,
            "Sheet2_TP_full_norm": s2_full,
            "Sheet2_TP_base": s2_base,
            "Sheet2_Histology": clean_text(s2.get("Histology", "")),
        })

matches_df = pd.DataFrame(match_records)
not_found_df = pd.DataFrame(not_found_records)

matches_df.to_csv(match_out, index=False)
not_found_df.to_csv(not_found_out, index=False)

# ===============================
# Summary
# ===============================
print("\n===============================")
print("TP match summary")
print("===============================")
print(f"Sheet2 TP entries checked: {len(sheet2_tp)}")
print(f"Matched Sheet2 TP entries: {matches_df['Sheet2_TP_raw'].nunique() if not matches_df.empty else 0}")
print(f"Not found Sheet2 TP entries: {len(not_found_df)}")

if not matches_df.empty:
    print("\nMatch type counts:")
    print(matches_df["Match_Type"].value_counts())

print("\nOutput files:")
print(f"  Matches:   {match_out}")
print(f"  Not found: {not_found_out}")
print(f"  Sheet3 TP long table: {sheet3_long_out}")

Found 14 TP Number columns in Sheet3:
  - TP Number
  - TP Number.1
  - TP Number.2
  - TP Number.3
  - TP Number.4
  - TP Number.5
  - TP Number.6
  - TP Number.7
  - TP Number.8
  - TP Number.9
  - TP Number.10
  - TP Number.11
  - TP Number.12
  - TP Number.13

Extracted 106 TP Number entries from Sheet3
Saved Sheet3 long TP table: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/TP_match_check/Sheet3_extracted_TPNumber_long.csv

TP match summary
Sheet2 TP entries checked: 85
Matched Sheet2 TP entries: 6
Not found Sheet2 TP entries: 79

Match type counts:
BASE_TP_MATCH_ONLY    7
Name: Match_Type, dtype: int64

Output files:
  Matches:   /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/TP_match_check/Sheet2_TP_matches_Sheet3_TPNumber.csv
  Not found: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/TP_match_check/Sheet2_TP_not_found_in_Sheet3.csv
  Sheet3 TP long table: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/TP_match_check/Sheet3_extracted_TPNumber_long.csv


In [2]:
# ===============================
# Print unique matched TP numbers
# ===============================
print("\n===============================")
print("Unique TP numbers matched between Sheet2 and Sheet3")
print("===============================")

if matches_df.empty:
    print("No TP numbers matched.")
else:
    unique_matches = (
        matches_df[
            [
                "Sheet2_TP_base",
                "Sheet2_TP_raw",
                "Sheet3_TP_raw",
                "Sheet3_REDCAP",
                "Sheet3_BIOS",
                "Sheet3_Histology",
            ]
        ]
        .drop_duplicates(subset=["Sheet2_TP_base"])
        .sort_values("Sheet2_TP_base")
    )

    for _, row in unique_matches.iterrows():
        print(
            f"{row['Sheet2_TP_base']}  "
            f"| Sheet2: {row['Sheet2_TP_raw']}  "
            f"| Sheet3: {row['Sheet3_TP_raw']}  "
            f"| REDCAP: {row['Sheet3_REDCAP']}  "
            f"| BIOS: {row['Sheet3_BIOS']}  "
            f"| Histology: {row['Sheet3_Histology']}"
        )

    print(f"\nTotal unique matched TP numbers: {len(unique_matches)}")


Unique TP numbers matched between Sheet2 and Sheet3
TP21-M131  | Sheet2: TP21-M131  | Sheet3: TP21-M131 (7)  | REDCAP: 4055  | BIOS: 68736  | Histology: IDC
TP21-M557  | Sheet2: TP21-M557  | Sheet3: TP21-M557 (4)  | REDCAP: 4442  | BIOS: 61527  | Histology: IDC
TP23-M200  | Sheet2: TP23-M200  | Sheet3: TP23-M200(8)  | REDCAP: 4597  | BIOS: 85995  | Histology: IDC
TP23-M242  | Sheet2: TP23-M242  | Sheet3: TP23-M242 (9)  | REDCAP: 4324  | BIOS: 43788  | Histology: IDC
TP23-M62  | Sheet2: TP23-M62  | Sheet3: TP23-M62 (5)  | REDCAP: 4272  | BIOS: 26044  | Histology: IDC
TP24-M292  | Sheet2: TP24-M292  | Sheet3: TP24-M292 (8)  | REDCAP: 4313  | BIOS: 44909  | Histology: IDC

Total unique matched TP numbers: 6
